Cell 1: Setup & Fungsi Ekstraksi (Poin i & ii)

Cell ini khusus untuk mendefinisikan Regular Expression (Regex) yang akan bertugas "memburu" metadata dan konten kunci dari dalam teks putusan.

In [4]:
import os
import re
import pandas as pd

# Konfigurasi Folder
folder_raw = "../data/raw"
folder_processed = "../data/processed"
os.makedirs(folder_processed, exist_ok=True)

# i. Ekstraksi Metadata & ii. Ekstraksi Konten Kunci
def ekstrak_informasi(teks):
    # 1. Nomor Perkara

    pola_no = r'(?:nomor|no)[\s\:\.]*([0-9]+\s*[a-z]*\s*\/\s*[a-z\.\-]+\s*\/(?:[a-z]+\s*\/)?\s*[0-9]{4}(?:\s*\/\s*[a-z\s\.]+)?)'
    match_no = re.search(pola_no, teks, re.IGNORECASE)
    no_perkara = re.sub(r'\s+', ' ', match_no.group(1)).upper().strip() if match_no else "TIDAK DITEMUKAN"

    # 2. Tanggal Putusan
    pola_tgl = r'tanggal\s+(\d{1,2}\s+[a-z]+\s+\d{4})'
    match_tgl = re.search(pola_tgl, teks, re.IGNORECASE)
    tanggal = match_tgl.group(1).strip() if match_tgl else "TIDAK DITEMUKAN"

    # 3. Pasal (Mencari penyebutan pasal terkait Merek/HKI)
    pola_pasal = r'(pasal\s+\d+[\s\w\-\d\,]+(?:merek|undang-undang|uu))'
    match_pasal = re.search(pola_pasal, teks, re.IGNORECASE)
    pasal = match_pasal.group(1).strip() if match_pasal else "TIDAK DITEMUKAN"

    # 4. Pihak Penggugat & Tergugat (Pendekatan sederhana mencari kata Penggugat dan Tergugat)
    # Teks hukum sangat variatif, kita ambil cuplikan sekitar kata tersebut
    pola_pihak = r'antara\s*:\s*(.*?)\s*sebagai\s*penggugat.*?(?:melawan|dan)\s*(.*?)\s*sebagai\s*tergugat'
    match_pihak = re.search(pola_pihak, teks, re.IGNORECASE | re.DOTALL)
    if match_pihak:
        penggugat = re.sub(r'\s+', ' ', match_pihak.group(1)).strip()[:50]
        tergugat = re.sub(r'\s+', ' ', match_pihak.group(2)).strip()[:50]
        pihak = f"{penggugat} VS {tergugat}"
    else:
        pihak = "TIDAK DITEMUKAN"

    # 5. Ringkasan Fakta (Mengambil teks setelah DUDUK PERKARA)
    pola_fakta = r'(?:duduk\s*perkara|tentang\s*duduk\s*perkara)(.*?)(?:menimbang|tentang\s*pertimbangan)'
    match_fakta = re.search(pola_fakta, teks, re.IGNORECASE | re.DOTALL)
    ringkasan_fakta = match_fakta.group(1).strip()[:1000] + "..." if match_fakta else "TIDAK DITEMUKAN"
    
    # 6. Argumen Hukum / Solusi (Amar Putusan setelah MENGADILI)
    pola_amar = r'M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*[\:\.]?\s*(.*)'
    match_amar = re.search(pola_amar, teks, re.IGNORECASE | re.DOTALL)
    amar_putusan = match_amar.group(1).strip()[:1000] + "..." if match_amar else "TIDAK DITEMUKAN"

    return no_perkara, tanggal, pasal, pihak, ringkasan_fakta, amar_putusan

print("Fungsi Ekstraksi berhasil di-load!")

Fungsi Ekstraksi berhasil di-load!


Cell 2: Eksekusi & Feature Engineering (Poin ii.1)

Di sini akan me-looping file, menjalankan ekstraksi, dan menambahkan Feature Engineering berupa menghitung length (jumlah kata) sesuai opsi di modul.

In [6]:
# Eksekusi Looping & ii.1. Feature Engineering
print("Memulai pembacaan dokumen dan feature engineering...")
data_kasus = []

for nama_file in os.listdir(folder_raw):
    if nama_file.endswith('.txt'):
        path_txt = os.path.join(folder_raw, nama_file)
        
        with open(path_txt, 'r', encoding='utf-8') as f:
            teks_bersih = f.read()
        
        # Eksekusi fungsi dari cell 1
        no_perkara, tanggal, pasal, pihak, ringkasan_fakta, amar_putusan = ekstrak_informasi(teks_bersih)
        
        # ii.1 Feature Engineering: Hitung length (jumlah kata)
        jumlah_kata = len(teks_bersih.split())
        
        # Memasukkan ke list dengan struktur kolom yang diminta modul
        data_kasus.append({
            'case_id': nama_file.replace('.txt', ''),
            'no_perkara': no_perkara,
            'tanggal': tanggal,
            'pasal': pasal,
            'pihak': pihak,
            'ringkasan_fakta': ringkasan_fakta,
            'amar_putusan': amar_putusan, 
            'jumlah_kata': jumlah_kata,   
            'text_full': teks_bersih
        })

# Jadikan DataFrame
df_cases = pd.DataFrame(data_kasus)

print(f"Selesai mengekstrak {len(df_cases)} kasus!")
# Menampilkan kolom persis seperti urutan contoh di tabel modul
display(df_cases[['case_id', 'no_perkara', 'tanggal', 'ringkasan_fakta', 'pasal', 'pihak', 'text_full']].head(2))

Memulai pembacaan dokumen dan feature engineering...
Selesai mengekstrak 30 kasus!


,case_id,no_perkara,tanggal,ringkasan_fakta,pasal,pihak,text_full
0,case_001,7/PDT.SUS-HKI/MEREK/2024/PN NIAGA SBY,18 september 2024,...,pasal 1 angka 5 undang-undang nomor 20 tahun 2...,"made yanie mason, jenis kelamin: perempuan, te...",hkama ahkamah agung repub ahkamah agung republ...
1,case_002,8/PDT.SUS-MEREK/2022/PN NIAGA JKT.PST DEMI KEA...,28 desember 2021,...,pasal 20 atau pasal 21 undang-undang nomor 20 ...,"bytedance ltd., suatu badan hukum yang didirik...",hkama ahkamah agung repub ahkamah agung republ...


Cell 3: Penyimpanan (Poin ii.2)

In [7]:
# ii.2. Penyimpanan (Simpan ke format terstruktur)

# Kita simpan ke dalam bentuk CSV
path_csv = os.path.join(folder_processed, "cases.csv")

# Menyimpan DataFrame ke CSV
df_cases.to_csv(path_csv, index=False)

print("-" * 60)
print(f"Tahap 2 Selesai! Data berhasil disimpan di: {path_csv}")
print("-" * 60)

# (Opsional) Jika ingin menyimpan ke JSON juga seperti instruksi
path_json = os.path.join(folder_processed, "cases.json")
df_cases.to_json(path_json, orient='records', indent=4)
print(f"Data juga dibackup ke JSON di: {path_json}")

------------------------------------------------------------
Tahap 2 Selesai! Data berhasil disimpan di: ../data/processed\cases.csv
------------------------------------------------------------
Data juga dibackup ke JSON di: ../data/processed\cases.json
